# ChartVLM — Classificação dos Crops dos Detectores

**Objetivo:** usar o ChartVLM-large como camada 2 do pipeline — classificar cada crop produzido pelos detectores anteriores em tipo de gráfico (bar, line, pie, scatter, etc.).

**Detectores que produziram crops:**

- **Chandra OCR 2** — figuras salvas como `.webp` em `output/chandra/<pdf_stem>/`
- **PyMuPDF raster** — imagens em `output/pymupdf_raster/<pdf_stem>/`

> O dots.ocr 1.5 foi originalmente planejado como terceiro detector, mas sua execução foi inviabilizada por incompatibilidades técnicas em Colab (vide §4.4 do relatório técnico da sprint).

**Pipeline:**

1. Coleta crops dos 2 detectores no Drive.
2. Roda 3 prompts no ChartVLM para cada crop (tipo, título, CSV de dados).
3. Salva CSV com checkpoint a cada 20 crops.
4. Avaliação manual via widget interativo.
5. Métricas finais: acurácia por detector de origem.

> ⚠️ **ChartVLM exige `transformers==4.31.0`** — incompatível com os demais notebooks. Reset do runtime antes/depois.

> ⚠️ **GPU forte:** ChartVLM-large tem ~13B parâmetros em FP16. Recomendado L4 (22 GB VRAM) ou A100. T4 funciona mas é lento.


## 1. Instalação — versões fixas do ChartVLM

In [ ]:
# IMPORTANTE: ChartVLM exige versões antigas. Reset do runtime se já tiver carregado outros modelos.
!pip install -q torch==2.1.0 torchvision==0.16.0 transformers==4.31.0 \
    accelerate==0.24.0 huggingface_hub einops sentencepiece protobuf timm \
    pandas matplotlib ipywidgets pillow numpy

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}", end='')
if torch.cuda.is_available():
    print(f"  |  {torch.cuda.get_device_name(0)}  |  "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM")
print(f"\ntorch: {torch.__version__}")
import transformers
print(f"transformers: {transformers.__version__}")

## 2. Clone do repo + download dos pesos

O ChartVLM tem 3 subcomponentes (base_decoder, auxiliary_decoder, instruction_adapter). A função `infer_ChartVLM` lida com o roteamento entre eles.

In [ ]:
import os
os.chdir('/content')

# Clone do repo (contém o código do infer_ChartVLM)
!rm -rf /content/ChartVLM
!git clone -q https://github.com/InternScience/ChartVLM.git /content/ChartVLM

# Instala dependências internas do repo (fire, gradio, etc.)
!pip install -q -r /content/ChartVLM/requirements.txt

import sys
sys.path.insert(0, '/content/ChartVLM')
print("✓ Repo clonado e dependências instaladas")

In [ ]:
# Download dos pesos via huggingface_hub
from huggingface_hub import snapshot_download
from pathlib import Path

CHARTVLM_WEIGHTS_DIR = Path('/content/chartvlm_weights')
CHARTVLM_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

print("Baixando pesos do ChartVLM-large (~24 GB, ~10-15 min na primeira vez)...")
snapshot_download(
    repo_id='ahmed-masry/chartvlm-large',
    local_dir=str(CHARTVLM_WEIGHTS_DIR),
    allow_patterns=['*.bin', '*.json', '*.txt', '*.model', 'tokenizer*']
)
print(f"✓ Pesos em {CHARTVLM_WEIGHTS_DIR}")

In [ ]:
# Verifica estrutura esperada das 3 subpastas
expected_subdirs = ['base_decoder', 'auxiliary_decoder', 'instruction_adapter']
for s in expected_subdirs:
    p = CHARTVLM_WEIGHTS_DIR / s
    if p.exists():
        files = list(p.iterdir())
        print(f"  ✓ {s}/: {len(files)} arquivos")
    else:
        print(f"  ✗ {s}/: NÃO EXISTE — verifique a estrutura do repo no HF")

## 3. Configuração de pastas + montagem do Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

OUTPUT_ROOT  = Path('/content/drive/MyDrive/Chandra2/output')
CHANDRA_DIR  = OUTPUT_ROOT / 'chandra'
RASTER_DIR   = OUTPUT_ROOT / 'pymupdf_raster'
CHARTVLM_OUT = OUTPUT_ROOT / 'chartvlm_results'
CHARTVLM_OUT.mkdir(parents=True, exist_ok=True)

OUT_CSV = CHARTVLM_OUT / 'chartvlm_classifications.csv'

print(f"Output: {CHARTVLM_OUT}")

## 4. Coletar todos os crops dos detectores

Crops do **Chandra** (`.webp`) e **PyMuPDF raster** (`.png`). Não inclui dots.ocr.

In [ ]:
import json
from PIL import Image

all_crops = []

# ----- Chandra: .webp/png/jpg em cada subpasta -----
for sub in CHANDRA_DIR.iterdir():
    if not sub.is_dir(): continue
    pdf_stem = sub.name
    for crop in sub.glob('*.webp'):
        all_crops.append({'pdf': pdf_stem, 'detector': 'chandra',
                          'crop_path': str(crop)})
    for crop in sub.glob('*.png'):
        all_crops.append({'pdf': pdf_stem, 'detector': 'chandra',
                          'crop_path': str(crop)})

# ----- PyMuPDF raster: .png em cada subpasta -----
for sub in RASTER_DIR.iterdir():
    if not sub.is_dir(): continue
    pdf_stem = sub.name
    for crop in sub.glob('*.png'):
        all_crops.append({'pdf': pdf_stem, 'detector': 'pymupdf_raster',
                          'crop_path': str(crop)})

from collections import Counter
det_count = Counter(c['detector'] for c in all_crops)
print(f"Total crops coletados: {len(all_crops)}")
for d, n in det_count.items():
    print(f"  {d}: {n}")

## 5. Carregar ChartVLM e definir helper de inferência

In [ ]:
# A função infer_ChartVLM cuida do roteamento de tarefas via instruction adapter
from tools.ChartVLM import infer_ChartVLM
import numpy as np
import time

CHARTVLM_MODEL_DIR = CHARTVLM_WEIGHTS_DIR

# Warmup com imagem dummy (primeira chamada carrega os 3 submodels)
warmup_img_path = '/tmp/warmup.png'
Image.fromarray(np.full((300, 300, 3), [200, 30, 30], dtype=np.uint8)).save(warmup_img_path)

print("Warmup do modelo (1-2 min na primeira vez)...")
t0 = time.time()
try:
    out = infer_ChartVLM(warmup_img_path, "What type of chart is this?", CHARTVLM_MODEL_DIR)
    print(f"✓ Modelo respondeu em {time.time()-t0:.1f}s")
    print(f"  resposta: {out}")
except Exception as e:
    print(f"✗ Erro no warmup: {e}")
    print("  → confira se as 3 subpastas existem em CHARTVLM_MODEL_DIR")

In [ ]:
# 3 prompts por crop — pulamos 'descricao' porque ChartVLM tem bug
# com load_in_8bit em transformers novos no modo descricao
PROMPTS = {
    'tipo':       "What type of chart is shown in the image?",
    'titulo':     "What is the title of the chart?",
    'csv':        "Convert this chart into a CSV table.",
}

def chartvlm_full(image_path):
    """Roda os prompts em sequência sobre a mesma imagem."""
    results = {'tipo': '', 'titulo': '', 'csv': '', 'descricao': ''}
    for key, prompt in PROMPTS.items():
        try:
            out = infer_ChartVLM(image_path, prompt, CHARTVLM_MODEL_DIR)
            if isinstance(out, dict):
                out = out.get('text', out.get('answer', str(out)))
            results[key] = str(out).strip()
        except Exception as e:
            results[key] = f"ERROR: {e}"
    return results

## 6. Inferência em todos os crops

Tempo estimado: ~10-15s por crop (3 prompts × ~5s). Para 170 crops: ~50 min.

**Retomada inteligente:** se o Colab desconectar no meio, basta rerodar esta célula — ela pula crops já processados.

In [ ]:
import pandas as pd, time, json

# ----- Retomada inteligente: se já existe CSV, pula crops já processados -----
if OUT_CSV.exists():
    existing = pd.read_csv(OUT_CSV)
    # Considera "feito" só se tipo_predito está preenchido (não vazio, não NaN)
    existing['tipo_predito'] = existing['tipo_predito'].fillna('').astype(str).str.strip()
    done_mask = existing['tipo_predito'] != ''
    done = set(existing[done_mask]['crop_path'].tolist())
    rows = existing[done_mask].to_dict('records')
    print(f"📂 CSV existe — {len(done)}/{len(existing)} crops válidos, retomando")
else:
    done = set()
    rows = []

to_process = [c for c in all_crops if c['crop_path'] not in done]
print(f"A processar agora: {len(to_process)}")

t_global = time.time()
for i, item in enumerate(to_process, 1):
    t0 = time.time()
    result = chartvlm_full(item['crop_path'])
    rows.append({
        'pdf':             item['pdf'],
        'detector':        item['detector'],
        'crop_path':       item['crop_path'],
        'crop_filename':   Path(item['crop_path']).name,
        'tipo_predito':    result['tipo'],
        'titulo':          result['titulo'],
        'csv_dados':       result['csv'],
        'descricao':       result['descricao'],
        'tipo_real':       '',
        'classificou_certo': '',
    })

    if i % 20 == 0 or i == len(to_process):
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
        elapsed = (time.time()-t_global)/60
        print(f"  [{i}/{len(to_process)}] checkpoint ({elapsed:.1f} min)")

    print(f"  [{i}/{len(to_process)}] {item['detector']:<15} "
          f"{Path(item['crop_path']).name[:30]:<32} → tipo: {result['tipo'][:30]}  ({time.time()-t0:.1f}s)")

pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f"\n=== Concluído em {(time.time()-t_global)/60:.1f} min ===")
print(f"CSV: {OUT_CSV}")

## 7. Avaliação manual via widget interativo

Para cada crop não avaliado, mostra a imagem + o que o ChartVLM previu, e você marca:

- **✓ SIM** = ChartVLM acertou o tipo
- **~ PARCIAL** = categoria certa mas subtipo errado (ex: classifica multi_line como line_chart)
- **✗ NÃO** = errou completamente OU o crop não é gráfico

Salva automaticamente a cada clique.

In [ ]:
import pandas as pd
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

df = pd.read_csv(OUT_CSV)

TIPOS_CANONICOS = [
    'bar_chart', 'multi_bar', 'line_chart', 'multi_line', 'pie_chart',
    'scatter', 'radar', 'histogram', 'multi_histogram',
    'heatmap', 'boxplot', 'area', 'funnel', 'treemap',
    'bubble', 'candlestick', '3D-Bar',
    'table', 'photo', 'diagram', 'logo',
    'text_fragment', 'not_a_chart',
]

def is_pending(row):
    return pd.isna(row['classificou_certo']) or str(row['classificou_certo']).strip() == ''

pending_idx = [i for i, row in df.iterrows() if is_pending(row)]
print(f"Total: {len(df)} crops  |  Pendentes: {len(pending_idx)}")

state = {'current_pos': 0, 'pending_idx': pending_idx}

img_out, info_out, status_out = widgets.Output(), widgets.Output(), widgets.Output()

tipo_dropdown = widgets.Dropdown(options=[''] + TIPOS_CANONICOS, description='tipo_real:',
                                  style={'description_width': '120px'},
                                  layout=widgets.Layout(width='350px'))
custom_tipo = widgets.Text(description='ou outro:', placeholder='digite tipo customizado',
                            style={'description_width': '120px'},
                            layout=widgets.Layout(width='350px'))

def mkbtn(label, color, value):
    b = widgets.Button(description=label, button_style=color,
                       layout=widgets.Layout(width='110px', height='40px'))
    b.value = value; return b

btn_sim     = mkbtn('✓ SIM',     'success', 'sim')
btn_parcial = mkbtn('~ PARCIAL', 'warning', 'parcial')
btn_nao     = mkbtn('✗ NÃO',     'danger',  'nao')
btn_skip    = mkbtn('→ pular',   '',        None)
btn_voltar  = mkbtn('← voltar',  '',        'back')

def render_current():
    img_out.clear_output(); info_out.clear_output(); status_out.clear_output()
    pos, pend = state['current_pos'], state['pending_idx']
    if pos >= len(pend):
        with status_out:
            print("✅ FIM — todos avaliados!")
        return
    idx = pend[pos]
    row = df.loc[idx]
    with img_out:
        try:
            img = Image.open(row['crop_path'])
            fig, ax = plt.subplots(figsize=(7, 6))
            ax.imshow(img); ax.axis('off')
            ax.set_title(f"[{pos+1}/{len(pend)}] {row['crop_filename']}", fontsize=10)
            plt.show()
        except Exception as e:
            print(f"Erro ao abrir: {e}")
    with info_out:
        print(f"PDF:           {row['pdf']}")
        print(f"Detector:      {row['detector']}")
        print(f"─────────────────────────────────")
        print(f"ChartVLM disse:")
        print(f"  tipo_predito: {row['tipo_predito']}")
        print(f"  titulo:       {str(row.get('titulo',''))[:80]}")
        print(f"  csv_dados:    {str(row.get('csv_dados',''))[:120]}")
    # Pré-preenche dropdown
    suggested = str(row.get('tipo_predito', '')).strip()
    if suggested in TIPOS_CANONICOS:
        tipo_dropdown.value = suggested; custom_tipo.value = ''
    else:
        tipo_dropdown.value = ''; custom_tipo.value = ''

def save_and_advance(eval_value):
    pos, pend = state['current_pos'], state['pending_idx']
    if pos >= len(pend): return
    idx = pend[pos]
    if eval_value is not None:
        tipo = custom_tipo.value.strip() if custom_tipo.value.strip() else tipo_dropdown.value
        if not tipo:
            with status_out:
                clear_output(); print("⚠ Selecione um tipo_real antes de avaliar.")
            return
        df.at[idx, 'tipo_real'] = tipo
        df.at[idx, 'classificou_certo'] = eval_value
        df.to_csv(OUT_CSV, index=False)
    state['current_pos'] = pos + 1
    render_current()

btn_sim.on_click(lambda _: save_and_advance('sim'))
btn_parcial.on_click(lambda _: save_and_advance('parcial'))
btn_nao.on_click(lambda _: save_and_advance('nao'))
btn_skip.on_click(lambda _: save_and_advance(None))
btn_voltar.on_click(lambda _: (state.update({'current_pos': max(0, state['current_pos']-1)}), render_current()))

print("=" * 60)
print("AVALIADOR DE CROPS — ChartVLM × Detector")
print("=" * 60)
display(img_out)
display(info_out)
display(widgets.HBox([tipo_dropdown, custom_tipo]))
display(widgets.HBox([btn_sim, btn_parcial, btn_nao, btn_skip, btn_voltar]))
display(status_out)
render_current()

## 8. Métricas finais — acurácia por detector

Roda **depois** que você preencher `classificou_certo` em uma quantidade significativa de crops (recomendado: pelo menos 50 por detector).

In [ ]:
import pandas as pd

df = pd.read_csv(OUT_CSV)

# Versão tolerante a NaN
def is_valid_eval(v):
    if pd.isna(v): return False
    return str(v).lower().strip() in {'sim','nao','parcial'}

avaliadas = df[df['classificou_certo'].apply(is_valid_eval)].copy()
print(f"Crops avaliados: {len(avaliadas)} / {len(df)}")

if len(avaliadas) == 0:
    print("\n⚠️ Nenhum crop avaliado ainda. Use o widget acima para preencher.")
else:
    score_map = {'sim': 1.0, 'parcial': 0.5, 'nao': 0.0}
    avaliadas['score'] = (avaliadas['classificou_certo']
                          .astype(str).str.lower().str.strip().map(score_map))

    print("\n=== ACURÁCIA POR DETECTOR ===\n")
    by_detector = avaliadas.groupby('detector').agg(
        n_crops=('score','count'),
        n_acerto=('score', lambda s: (s==1.0).sum()),
        n_parcial=('score', lambda s: (s==0.5).sum()),
        n_erro=('score', lambda s: (s==0.0).sum()),
        acuracia_strict=('score', lambda s: (s==1.0).mean()),
        acuracia_lenient=('score','mean'),
    ).sort_values('acuracia_strict', ascending=False)
    print(by_detector.round(3).to_string())

    if avaliadas['tipo_real'].notna().any():
        print("\n=== ACURÁCIA POR DETECTOR × TIPO DE GRÁFICO ===\n")
        ct = avaliadas.groupby(['detector', 'tipo_real']).agg(
            n=('score','count'),
            acerto=('score', lambda s: (s==1.0).mean()),
        ).round(2)
        print(ct.to_string())

    by_detector.to_csv(CHARTVLM_OUT / 'acuracia_por_detector.csv')
    print(f"\n💾 Salvo: {CHARTVLM_OUT/'acuracia_por_detector.csv'}")

    winner = by_detector['acuracia_strict'].idxmax()
    print(f"\n🏆 Detector que produziu crops mais classificáveis pelo ChartVLM:")
    print(f"   {winner} — strict {by_detector.loc[winner,'acuracia_strict']*100:.1f}%, "
          f"lenient {by_detector.loc[winner,'acuracia_lenient']*100:.1f}%")

## 9. Conclusão e próximos passos

**Achados observados na sprint (29/abr–13/mai 2026):**

- ChartVLM-large atingiu apenas 19,4% strict (50,0% lenient) sobre crops do Chandra, e 14,3% strict (31,6% lenient) sobre crops do PyMuPDF raster.
- Falha catastrófica em scatter: 0% de acertos em 16 crops de dispersão (modelo classifica-os como bubble ou heatmap).
- Modelo não tem classe `not_a_chart`, então força encaixe em alguma categoria; crops de logos e fotos viram funnel, treemap, etc.

**Limitação técnica observada:**

O prompt original `What's the description of the chart?` retornava `ERROR: LlamaForCausalLM.__init__() got an unexpected keyword argument 'load_in_8bit'` em todas as inferências, devido à mudança no Transformers ≥ 4.30. A solução adotada foi remover esse prompt do pipeline. Os campos `tipo`, `titulo` e `csv_dados` foram preservados.

**Comparação com LlamaParse** (notebook N4): nas mesmas condições e crops, o LlamaParse atingiu 91,7% lenient sobre crops do Chandra — atendendo ao critério de 90% da sprint.